In [1]:
# Transilien BI Project

## Objective
"""
Analyze train punctuality and identify influencing factors such as:
- weather
- strikes
- holidays
- temporal patterns
"""

## Pipeline
"""
Raw data → Cleaning → Enrichment → Export for SAP Analytics Cloud
"""

'\nRaw data → Cleaning → Enrichment → Export for SAP Analytics Cloud\n'

In [2]:
## Import libraries

In [3]:

"""
We import the libraries required for:
- data manipulation
- feature engineering
- holiday generation
"""
import pandas as pd
import holidays
import matplotlib.pyplot as plt
import numpy as np

In [4]:
## Load the Transilien dataset

In [5]:
"""
The original dataset contains monthly punctuality indicators for Transilien lines.
"""
df = pd.read_csv("../Data/raw/ponctualite-mensuelle-transilien.csv",sep=";")
df.head()

,Date,Service,Ligne,Nom de la ligne,Taux de ponctualité,Nombre de voyageurs à l'heure pour un voyageur en retard
0,2013-01,RER,A,RER A,83.6,5.1
1,2013-01,Transilien,R,Paris Sud Est,87.2,6.8
2,2013-03,Transilien,H,Paris Nord Ouest,92.3,12.0
3,2013-04,Transilien,N,Paris Montparnasse,90.2,9.2
4,2013-05,RER,D,RER D,87.1,6.8


In [6]:
## Data cleaning

In [7]:
"""
We standardize column names and convert data types to prepare the dataset for analysis.
"""
df.columns =(
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("'","_")
    .str.replace("é","e")

    )
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2009 entries, 0 to 2008
Data columns (total 6 columns):
 #   Column                                                    Non-Null Count  Dtype  
---  ------                                                    --------------  -----  
 0   date                                                      2009 non-null   str    
 1   service                                                   2009 non-null   str    
 2   ligne                                                     2009 non-null   str    
 3   nom_de_la_ligne                                           2009 non-null   str    
 4   taux_de_ponctualite                                       2008 non-null   float64
 5   nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard  2009 non-null   float64
dtypes: float64(2), str(4)
memory usage: 94.3 KB


In [8]:
"""
We convert the date column into datetime format
and remove invalid date values.
"""
df["date"]=pd.to_datetime(df["date"])
df = df.dropna(subset="date")
df.info()



<class 'pandas.DataFrame'>
RangeIndex: 2009 entries, 0 to 2008
Data columns (total 6 columns):
 #   Column                                                    Non-Null Count  Dtype         
---  ------                                                    --------------  -----         
 0   date                                                      2009 non-null   datetime64[us]
 1   service                                                   2009 non-null   str           
 2   ligne                                                     2009 non-null   str           
 3   nom_de_la_ligne                                           2009 non-null   str           
 4   taux_de_ponctualite                                       2008 non-null   float64       
 5   nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard  2009 non-null   float64       
dtypes: datetime64[us](1), float64(2), str(3)
memory usage: 94.3 KB


In [9]:
## Feature engineering

In [10]:
"""
We create temporal variables and KPIs to improve BI analysis.
"""
df["annee"]=df["date"].dt.year
df["mois"]=df["date"].dt.month
df["nom_mois"]=df["date"].dt.month_name()
df["trimestre"]=df["date"].dt.quarter
df["taux_irregularite"] = 100 - df["taux_de_ponctualite"]

df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 2009 entries, 0 to 2008
Data columns (total 11 columns):
 #   Column                                                    Non-Null Count  Dtype         
---  ------                                                    --------------  -----         
 0   date                                                      2009 non-null   datetime64[us]
 1   service                                                   2009 non-null   str           
 2   ligne                                                     2009 non-null   str           
 3   nom_de_la_ligne                                           2009 non-null   str           
 4   taux_de_ponctualite                                       2008 non-null   float64       
 5   nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard  2009 non-null   float64       
 6   annee                                                     2009 non-null   int32         
 7   mois                                                 

,date,service,ligne,nom_de_la_ligne,taux_de_ponctualite,nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard,annee,mois,nom_mois,trimestre,taux_irregularite
0,2013-01-01,RER,A,RER A,83.6,5.1,2013,1,January,1,16.4
1,2013-01-01,Transilien,R,Paris Sud Est,87.2,6.8,2013,1,January,1,12.8
2,2013-03-01,Transilien,H,Paris Nord Ouest,92.3,12.0,2013,3,March,1,7.7
3,2013-04-01,Transilien,N,Paris Montparnasse,90.2,9.2,2013,4,April,2,9.8
4,2013-05-01,RER,D,RER D,87.1,6.8,2013,5,May,2,12.9


In [11]:
## Holiday enrichment

In [12]:
"""
French holidays are generated dynamically using the holidays Python package.
"""
fr_holidays = holidays.France(years=df["annee"].unique()) 
"""
We create a function that calculates the number of French holidays
for a month passed as a parameter.
"""
def count_holidays_in_month(date):
    #first day of the month
    start = date.replace(day=1)
    #last day of the month using MonthEnd offset
    end = start + pd.offsets.MonthEnd(0)
    #Generate all days in the month
    days = pd.date_range(start, end, freq="D")
    #Count how many of these days are in the list of French holidays
    return sum(day.date() in fr_holidays for day in days)




In [13]:
"""We apply the function to the date column 
to create a new variable that counts the number
 of holidays in each month."""

df["nb_jours_feries_mois"] = df["date"].apply(count_holidays_in_month)
df.head()

,date,service,ligne,nom_de_la_ligne,taux_de_ponctualite,nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard,annee,mois,nom_mois,trimestre,taux_irregularite,nb_jours_feries_mois
0,2013-01-01,RER,A,RER A,83.6,5.1,2013,1,January,1,16.4,1
1,2013-01-01,Transilien,R,Paris Sud Est,87.2,6.8,2013,1,January,1,12.8,1
2,2013-03-01,Transilien,H,Paris Nord Ouest,92.3,12.0,2013,3,March,1,7.7,0
3,2013-04-01,Transilien,N,Paris Montparnasse,90.2,9.2,2013,4,April,2,9.8,1
4,2013-05-01,RER,D,RER D,87.1,6.8,2013,5,May,2,12.9,4


In [14]:
## Holiday feature validation

In [15]:
# Check that the new feature was created
print(df.columns)

# Display holiday counts for several months
print(df[
    ["date", "nom_mois", "nb_jours_feries_mois"]
].head(20))

# Verify there are no missing values
print(df["nb_jours_feries_mois"].isna().sum())

# Analyze holiday count distribution
df["nb_jours_feries_mois"].value_counts()

Index(['date', 'service', 'ligne', 'nom_de_la_ligne', 'taux_de_ponctualite',
       'nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard', 'annee',
       'mois', 'nom_mois', 'trimestre', 'taux_irregularite',
       'nb_jours_feries_mois'],
      dtype='str')
         date   nom_mois  nb_jours_feries_mois
0  2013-01-01    January                     1
1  2013-01-01    January                     1
2  2013-03-01      March                     0
3  2013-04-01      April                     1
4  2013-05-01        May                     4
5  2013-05-01        May                     4
6  2013-06-01       June                     0
7  2013-06-01       June                     0
8  2013-08-01     August                     1
9  2013-09-01  September                     0
10 2013-09-01  September                     0
11 2013-10-01    October                     0
12 2013-10-01    October                     0
13 2013-11-01   November                     2
14 2013-11-01   November      

nb_jours_feries_mois
1    894
0    791
2    169
4     91
3     64
Name: count, dtype: int64

In [16]:
## Seasonal feature engineering

In [17]:
"""
We create a function to determine the season
based on the month extracted from the date.
"""
def get_season(month):
    if month in [12, 1, 2]:
        return "Hiver"
    elif month in [3, 4, 5]:
        return "Printemps"
    elif month in [6, 7, 8]:
        return "Été"
    else:
        return "Automne"


In [18]:
"""
We create a seasonal variable to improve
temporal and weather-related analysis.
"""
df["saison"] = df["mois"].apply(get_season)

In [19]:
## Seasonal feature validation

In [20]:
# Display unique month-season combinations
# to verify the season mapping
print(
    df[
        ["mois", "nom_mois", "saison"]
    ]
    .drop_duplicates()
    .sort_values("mois")
)

# Analyze season distribution
print(
    df["saison"]
    .value_counts()
)

# Verify there are no missing values
print(
    df["saison"]
    .isna()
    .sum()
)

# Display the first rows of the dataset
# to confirm the new feature
df[
    ["date", "nom_mois", "saison"]
].head(20)

    mois   nom_mois     saison
0      1    January      Hiver
78     2   February      Hiver
2      3      March  Printemps
3      4      April  Printemps
4      5        May  Printemps
6      6       June        Été
33     7       July        Été
8      8     August        Été
9      9  September    Automne
11    10    October    Automne
13    11   November    Automne
17    12   December      Hiver
saison
Automne      507
Été          506
Hiver        505
Printemps    491
Name: count, dtype: int64
0


,date,nom_mois,saison
0,2013-01-01,January,Hiver
1,2013-01-01,January,Hiver
2,2013-03-01,March,Printemps
3,2013-04-01,April,Printemps
4,2013-05-01,May,Printemps
5,2013-05-01,May,Printemps
6,2013-06-01,June,Été
7,2013-06-01,June,Été
8,2013-08-01,August,Été
9,2013-09-01,September,Automne


In [21]:
## Weather enrichment

In [22]:
"""
We load the raw weather dataset from Météo-France.
This dataset contains daily weather observations for Paris.
"""
meteo = pd.read_csv("../Data/raw/meteo_gouv_1950-2024.csv", sep=";")
print(meteo.head())

   NUM_POSTE  NOM_USUEL        LAT       LON  ALTI  AAAAMMJJ   RR  QRR  TN  \
0   75101001  INNOCENTS  48.860667  2.348333    37  19500101  0.0  1.0 NaN   
1   75101001  INNOCENTS  48.860667  2.348333    37  19500102  1.8  1.0 NaN   
2   75101001  INNOCENTS  48.860667  2.348333    37  19500103  2.0  1.0 NaN   
3   75101001  INNOCENTS  48.860667  2.348333    37  19500104  0.2  1.0 NaN   
4   75101001  INNOCENTS  48.860667  2.348333    37  19500105  1.0  1.0 NaN   

   QTN  ...  FXI3S  QFXI3S  DXI3S  QDXI3S  HXI3S  QHXI3S  DRR  QDRR  \
0  NaN  ...    NaN     NaN    NaN     NaN    NaN     NaN  NaN   NaN   
1  NaN  ...    NaN     NaN    NaN     NaN    NaN     NaN  NaN   NaN   
2  NaN  ...    NaN     NaN    NaN     NaN    NaN     NaN  NaN   NaN   
3  NaN  ...    NaN     NaN    NaN     NaN    NaN     NaN  NaN   NaN   
4  NaN  ...    NaN     NaN    NaN     NaN    NaN     NaN  NaN   NaN   

   STATUS_FXI3S  STATUS_DXI3S  
0           NaN           NaN  
1           NaN           NaN  
2       

In [23]:
## Weather dataset exploration

In [24]:
# Display available weather columns
print(meteo.columns)
#Display dataset info to check data types and missing values
meteo.info()

Index(['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON', 'ALTI', 'AAAAMMJJ', 'RR', 'QRR',
       'TN', 'QTN', 'HTN', 'QHTN', 'TX', 'QTX', 'HTX', 'QHTX', 'TM', 'QTM',
       'TNTXM', 'QTNTXM', 'TAMPLI', 'QTAMPLI', 'TNSOL', 'QTNSOL', 'TN50',
       'QTN50', 'DG', 'QDG', 'FFM', 'QFFM', 'FF2M', 'QFF2M', 'FXY', 'QFXY',
       'DXY', 'QDXY', 'HXY', 'QHXY', 'FXI', 'QFXI', 'DXI', 'QDXI', 'HXI',
       'QHXI', 'FXI2', 'QFXI2', 'DXI2', 'QDXI2', 'HXI2', 'QHXI2', 'FXI3S',
       'QFXI3S', 'DXI3S', 'QDXI3S', 'HXI3S', 'QHXI3S', 'DRR', 'QDRR',
       'STATUS_FXI3S', 'STATUS_DXI3S'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 496112 entries, 0 to 496111
Data columns (total 60 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   NUM_POSTE     496112 non-null  int64  
 1   NOM_USUEL     496112 non-null  str    
 2   LAT           496112 non-null  float64
 3   LON           496112 non-null  float64
 4   ALTI          496112 non-null  int64  
 5   AAA

In [25]:
## Weather variable selection

In [26]:
"""
We keep only the weather variables required for the BI analysis:
- date
- rainfall
- average temperature
- average wind speed
"""
meteo = meteo[
    [
        "AAAAMMJJ",
        "RR",
        "TM",
        "FFM"
    ]
]

meteo.head()

,AAAAMMJJ,RR,TM,FFM
0,19500101,0.0,NaN,NaN
1,19500102,1.8,NaN,NaN
2,19500103,2.0,NaN,NaN
3,19500104,0.2,NaN,NaN
4,19500105,1.0,NaN,NaN


In [27]:
"""
We rename technical weather columns into clearer business-friendly names.
"""

meteo.columns = [
    "date",
    "pluie_mm",
    "temperature_moyenne",
    "vent_moyen"
]

meteo.head()

,date,pluie_mm,temperature_moyenne,vent_moyen
0,19500101,0.0,NaN,NaN
1,19500102,1.8,NaN,NaN
2,19500103,2.0,NaN,NaN
3,19500104,0.2,NaN,NaN
4,19500105,1.0,NaN,NaN


In [28]:
## Weather date conversion

In [29]:
"""
We convert the weather date column from YYYYMMDD format
to a proper datetime format.
"""

meteo["date"] = pd.to_datetime(
    meteo["date"],
    format="%Y%m%d",
    errors="coerce"
)

meteo.info()

<class 'pandas.DataFrame'>
RangeIndex: 496112 entries, 0 to 496111
Data columns (total 4 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   date                 496112 non-null  datetime64[us]
 1   pluie_mm             481784 non-null  float64       
 2   temperature_moyenne  62506 non-null   float64       
 3   vent_moyen           43957 non-null   float64       
dtypes: datetime64[us](1), float64(3)
memory usage: 15.1 MB


In [30]:
## Weather temporal filtering

In [31]:
"""
We keep weather observations starting from 2013
to match the Transilien dataset period.
"""

meteo = meteo[
    (meteo["date"] >= "2013-01-01") 
    
]

meteo.head()


,date,pluie_mm,temperature_moyenne,vent_moyen
53297,2013-01-01,0.8,NaN,NaN
53298,2013-01-02,1.3,NaN,NaN
53299,2013-01-03,0.2,NaN,NaN
53300,2013-01-04,0.0,NaN,NaN
53301,2013-01-05,0.0,NaN,NaN


In [32]:
## Weather date coverage validation

In [33]:
# Verify weather date coverage
print(meteo["date"].min())
print(meteo["date"].max())

# Analyze yearly weather observations
print(
    meteo["date"]
    .dt.year
    .value_counts()
    .sort_index()
)

2013-01-01 00:00:00
2024-12-31 00:00:00
date
2013    5382
2014    5147
2015    4076
2016    3669
2017    3984
2018    3987
2019    3886
2020    3875
2021    3904
2022    3375
2023    2252
2024    2196
Name: count, dtype: int64


In [34]:
## Weather preprocessing refinement

In [35]:
"""
An initial preprocessing pipeline was created for the Météo-France dataset.

Further exploration revealed that the dataset contained observations
from multiple weather stations with inconsistent temporal coverage.

Some stations only covered a limited time range,
while others contained observations extending until 2024.
This initially led to misleading preprocessing results
and large amounts of missing values after filtering and aggregation.

The weather preprocessing pipeline was therefore refined
to identify and select a single consistent weather station
covering the full analysis period (2013–2024).
"""

'\nAn initial preprocessing pipeline was created for the Météo-France dataset.\n\nFurther exploration revealed that the dataset contained observations\nfrom multiple weather stations with inconsistent temporal coverage.\n\nSome stations only covered a limited time range,\nwhile others contained observations extending until 2024.\nThis initially led to misleading preprocessing results\nand large amounts of missing values after filtering and aggregation.\n\nThe weather preprocessing pipeline was therefore refined\nto identify and select a single consistent weather station\ncovering the full analysis period (2013–2024).\n'

In [36]:
## Weather enrichment

In [37]:
"""
We load the raw Météo-France weather dataset.
"""

meteo = pd.read_csv(
    "../Data/raw/meteo_gouv_1950-2024.csv",
    sep=";"
)

meteo.head()

,NUM_POSTE,NOM_USUEL,LAT,LON,ALTI,AAAAMMJJ,RR,QRR,TN,QTN,...,FXI3S,QFXI3S,DXI3S,QDXI3S,HXI3S,QHXI3S,DRR,QDRR,STATUS_FXI3S,STATUS_DXI3S
0,75101001,INNOCENTS,48.860667,2.348333,37,19500101,0.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,75101001,INNOCENTS,48.860667,2.348333,37,19500102,1.8,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,75101001,INNOCENTS,48.860667,2.348333,37,19500103,2.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,75101001,INNOCENTS,48.860667,2.348333,37,19500104,0.2,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,75101001,INNOCENTS,48.860667,2.348333,37,19500105,1.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
## Weather variable selection

In [39]:
"""
We keep the weather variables required for the analysis,
including station information for validation purposes.
"""

meteo = meteo[
    [
        "NUM_POSTE",
        "NOM_USUEL",
        "AAAAMMJJ",
        "RR",
        "TM",
        "FFM"
    ]
]

meteo.head()

,NUM_POSTE,NOM_USUEL,AAAAMMJJ,RR,TM,FFM
0,75101001,INNOCENTS,19500101,0.0,NaN,NaN
1,75101001,INNOCENTS,19500102,1.8,NaN,NaN
2,75101001,INNOCENTS,19500103,2.0,NaN,NaN
3,75101001,INNOCENTS,19500104,0.2,NaN,NaN
4,75101001,INNOCENTS,19500105,1.0,NaN,NaN


In [40]:
## Weather date conversion

In [41]:
"""
We convert the weather date column into datetime format.
"""

meteo["AAAAMMJJ"] = pd.to_datetime(
    meteo["AAAAMMJJ"],
    format="%Y%m%d",
    errors="coerce"
)

meteo.info()

<class 'pandas.DataFrame'>
RangeIndex: 496112 entries, 0 to 496111
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   NUM_POSTE  496112 non-null  int64         
 1   NOM_USUEL  496112 non-null  str           
 2   AAAAMMJJ   496112 non-null  datetime64[us]
 3   RR         481784 non-null  float64       
 4   TM         62506 non-null   float64       
 5   FFM        43957 non-null   float64       
dtypes: datetime64[us](1), float64(3), int64(1), str(1)
memory usage: 22.7 MB


In [42]:
## Weather temporal filtering

In [43]:
"""
We keep weather observations starting from 2013
to match the Transilien dataset period.
"""

meteo = meteo[
    meteo["AAAAMMJJ"] >= "2013-01-01"
]

meteo.head()

,NUM_POSTE,NOM_USUEL,AAAAMMJJ,RR,TM,FFM
53297,75106001,LUXEMBOURG,2013-01-01,0.8,NaN,NaN
53298,75106001,LUXEMBOURG,2013-01-02,1.3,NaN,NaN
53299,75106001,LUXEMBOURG,2013-01-03,0.2,NaN,NaN
53300,75106001,LUXEMBOURG,2013-01-04,0.0,NaN,NaN
53301,75106001,LUXEMBOURG,2013-01-05,0.0,NaN,NaN


In [44]:
"""
We rename technical weather columns into clearer business-friendly names.
"""

meteo.columns = [
    "num_poste",
    "nom_usuel",
    "date",
    "pluie_mm",
    "temperature_moyenne",
    "vent_moyen"
]

meteo.head()

,num_poste,nom_usuel,date,pluie_mm,temperature_moyenne,vent_moyen
53297,75106001,LUXEMBOURG,2013-01-01,0.8,NaN,NaN
53298,75106001,LUXEMBOURG,2013-01-02,1.3,NaN,NaN
53299,75106001,LUXEMBOURG,2013-01-03,0.2,NaN,NaN
53300,75106001,LUXEMBOURG,2013-01-04,0.0,NaN,NaN
53301,75106001,LUXEMBOURG,2013-01-05,0.0,NaN,NaN


In [45]:
## Weather date coverage validation

In [46]:
# Verify weather date coverage
print(meteo["date"].min())
print(meteo["date"].max())

# Analyze yearly weather observations
print(
    meteo["date"]
    .dt.year
    .value_counts()
    .sort_index()
)

2013-01-01 00:00:00
2024-12-31 00:00:00
date
2013    5382
2014    5147
2015    4076
2016    3669
2017    3984
2018    3987
2019    3886
2020    3875
2021    3904
2022    3375
2023    2252
2024    2196
Name: count, dtype: int64


In [47]:
"""
Temperature availability varies significantly between weather stations.

Some stations provide observations until 2024,
while others contain large amounts of missing temperature data.
"""

'\nTemperature availability varies significantly between weather stations.\n\nSome stations provide observations until 2024,\nwhile others contain large amounts of missing temperature data.\n'

In [48]:
## Weather station coverage analysis

In [49]:
station_coverage = (
    meteo
    .groupby("nom_usuel")["date"]
    .agg(["min", "max", "count"])
    .sort_values("max", ascending=False)
)

station_coverage

,min,max,count
nom_usuel,,,
LONGCHAMP,2013-01-01,2024-12-31,4366
LUXEMBOURG,2013-01-01,2024-12-31,4263
TOUR EIFFEL,2013-01-01,2024-12-31,4217
PARIS-MONTSOURIS-DOUBLE,2016-08-25,2024-12-31,3051
LARIBOISIERE,2013-01-01,2024-12-31,4353
PARIS-MONTSOURIS,2013-01-01,2024-12-31,4383
BUTTES CHAUMONT,2013-01-01,2023-01-31,3622
ST-ANTOINE,2013-01-01,2023-01-31,3406
SALPETRIERE,2013-01-01,2022-12-31,3621


In [50]:
## Final weather station selection

In [51]:
"""
We select the PARIS-MONTSOURIS weather station
because it provides consistent weather observations
across the full analysis period (2013–2024).
"""

meteo = meteo[
    meteo["nom_usuel"] == "PARIS-MONTSOURIS"
]

print(meteo.head())
meteo.info()

        num_poste         nom_usuel       date  pluie_mm  temperature_moyenne  \
271362   75114001  PARIS-MONTSOURIS 2013-01-01       1.8                  7.6   
271363   75114001  PARIS-MONTSOURIS 2013-01-02       0.8                  6.6   
271364   75114001  PARIS-MONTSOURIS 2013-01-03       0.2                  9.7   
271365   75114001  PARIS-MONTSOURIS 2013-01-04       0.0                  9.8   
271366   75114001  PARIS-MONTSOURIS 2013-01-05       0.0                  9.0   

        vent_moyen  
271362         3.1  
271363         2.4  
271364         2.5  
271365         2.2  
271366         2.2  
<class 'pandas.DataFrame'>
Index: 4383 entries, 271362 to 275744
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   num_poste            4383 non-null   int64         
 1   nom_usuel            4383 non-null   str           
 2   date                 4383 non-null   datetime64[us]
 3   

In [52]:
## Weather station cleanup

In [53]:
"""
We remove weather station metadata
after selecting the final reference station.
"""

meteo = meteo.drop(
    columns=[
        "num_poste",
        "nom_usuel"
    ]
)

meteo.head()

,date,pluie_mm,temperature_moyenne,vent_moyen
271362,2013-01-01,1.8,7.6,3.1
271363,2013-01-02,0.8,6.6,2.4
271364,2013-01-03,0.2,9.7,2.5
271365,2013-01-04,0.0,9.8,2.2
271366,2013-01-05,0.0,9.0,2.2


In [54]:
## Weather feature validation

In [55]:
"""
We verify the completeness of the selected weather station
before continuing the preprocessing pipeline.
"""

meteo[["pluie_mm", "temperature_moyenne", "vent_moyen"]].isna().sum()

pluie_mm                0
temperature_moyenne     0
vent_moyen             15
dtype: int64

In [56]:
"""
The selected weather station shows a high level of data completeness,
with only a small number of missing wind observations remaining.
"""

'\nThe selected weather station shows a high level of data completeness,\nwith only a small number of missing wind observations remaining.\n'

In [57]:
## Monthly weather aggregation

In [58]:
"""
We create monthly time keys to aggregate daily weather observations
at the same granularity as the Transilien dataset.
"""

meteo["annee"] = meteo["date"].dt.year
meteo["mois"] = meteo["date"].dt.month

meteo.head()

,date,pluie_mm,temperature_moyenne,vent_moyen,annee,mois
271362,2013-01-01,1.8,7.6,3.1,2013,1
271363,2013-01-02,0.8,6.6,2.4,2013,1
271364,2013-01-03,0.2,9.7,2.5,2013,1
271365,2013-01-04,0.0,9.8,2.2,2013,1
271366,2013-01-05,0.0,9.0,2.2,2013,1


In [59]:
"""
We create monthly time keys to aggregate daily weather observations
at the same granularity as the Transilien dataset.
"""

meteo["annee"] = meteo["date"].dt.year
meteo["mois"] = meteo["date"].dt.month

meteo.head()

,date,pluie_mm,temperature_moyenne,vent_moyen,annee,mois
271362,2013-01-01,1.8,7.6,3.1,2013,1
271363,2013-01-02,0.8,6.6,2.4,2013,1
271364,2013-01-03,0.2,9.7,2.5,2013,1
271365,2013-01-04,0.0,9.8,2.2,2013,1
271366,2013-01-05,0.0,9.0,2.2,2013,1


In [60]:
"""
We aggregate daily weather observations at monthly level.

Temperature and wind are averaged over the month,
while rainfall is summed to represent total monthly precipitation.
"""

meteo_mensuelle = (
    meteo
    .groupby(["annee", "mois"], as_index=False)
    .agg({
        "temperature_moyenne": "mean",
        "pluie_mm": "sum",
        "vent_moyen": "mean"
    })
)

meteo_mensuelle.head(20)

,annee,mois,temperature_moyenne,pluie_mm,vent_moyen
0,2013,1,4.174194,42.2,2.800000
1,2013,2,3.321429,37.4,3.492857
2,2013,3,5.545161,36.6,3.106452
3,2013,4,11.040000,29.7,3.363333
4,2013,5,12.587097,111.2,2.861290
5,2013,6,17.370000,64.0,3.026667
6,2013,7,22.719355,47.8,2.993548
7,2013,8,20.554839,37.6,2.438710
8,2013,9,17.213333,49.5,2.280000
9,2013,10,14.245161,43.4,2.806452


In [61]:
## Weather and Transilien merge

In [62]:

df.head()

,date,service,ligne,nom_de_la_ligne,taux_de_ponctualite,nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard,annee,mois,nom_mois,trimestre,taux_irregularite,nb_jours_feries_mois,saison
0,2013-01-01,RER,A,RER A,83.6,5.1,2013,1,January,1,16.4,1,Hiver
1,2013-01-01,Transilien,R,Paris Sud Est,87.2,6.8,2013,1,January,1,12.8,1,Hiver
2,2013-03-01,Transilien,H,Paris Nord Ouest,92.3,12.0,2013,3,March,1,7.7,0,Printemps
3,2013-04-01,Transilien,N,Paris Montparnasse,90.2,9.2,2013,4,April,2,9.8,1,Printemps
4,2013-05-01,RER,D,RER D,87.1,6.8,2013,5,May,2,12.9,4,Printemps


In [63]:
"""
We merge monthly weather indicators
with the Transilien dataset.
"""

df = df.merge(
    meteo_mensuelle,
    on=["annee", "mois"],
    how="left"
)

df.head()

,date,service,ligne,nom_de_la_ligne,taux_de_ponctualite,nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard,annee,mois,nom_mois,trimestre,taux_irregularite,nb_jours_feries_mois,saison,temperature_moyenne,pluie_mm,vent_moyen
0,2013-01-01,RER,A,RER A,83.6,5.1,2013,1,January,1,16.4,1,Hiver,4.174194,42.2,2.800000
1,2013-01-01,Transilien,R,Paris Sud Est,87.2,6.8,2013,1,January,1,12.8,1,Hiver,4.174194,42.2,2.800000
2,2013-03-01,Transilien,H,Paris Nord Ouest,92.3,12.0,2013,3,March,1,7.7,0,Printemps,5.545161,36.6,3.106452
3,2013-04-01,Transilien,N,Paris Montparnasse,90.2,9.2,2013,4,April,2,9.8,1,Printemps,11.040000,29.7,3.363333
4,2013-05-01,RER,D,RER D,87.1,6.8,2013,5,May,2,12.9,4,Printemps,12.587097,111.2,2.861290


In [64]:
## Weather feature validation

In [65]:
""" Check missing values after weather merge""" 
df[
    [
        "temperature_moyenne",
        "pluie_mm",
        "vent_moyen"
    ]
].isna().sum()

temperature_moyenne    195
pluie_mm               195
vent_moyen             195
dtype: int64

In [66]:
""" Check missing values after weather merge by year"""
df.groupby("annee")[
    [
        "temperature_moyenne",
        "pluie_mm",
        "vent_moyen"
    ]
].apply(lambda x: x.isna().sum())

,temperature_moyenne,pluie_mm,vent_moyen
annee,,,
2013,0,0,0
2014,0,0,0
2015,0,0,0
2016,0,0,0
2017,0,0,0
2018,0,0,0
2019,0,0,0
2020,0,0,0
2021,0,0,0


In [67]:
"""
The weather merge was successfully validated.

Missing values are primarily associated with observations
outside the temporal coverage of the weather dataset,
confirming the consistency of the merge process.
"""

'\nThe weather merge was successfully validated.\n\nMissing values are primarily associated with observations\noutside the temporal coverage of the weather dataset,\nconfirming the consistency of the merge process.\n'

In [68]:
"""
We remove observations from 2025 and 2026
to ensure temporal consistency
with the weather dataset coverage.
"""
df = df[
    (df["annee"] != 2025)
    &
    (df["annee"] != 2026)
]
print((df["annee"] >= 2025).sum())

0


In [69]:
## Strike enrichment

In [70]:
"""
Strike dates were manually compiled using public SNCF strike records
and historical railway strike timelines available on Wikipedia:
https://wikimonde.com/article/Liste_des_gr%C3%A8ves_%C3%A0_la_SNCF

An official structured dataset containing daily strike observations
for the required analysis period was not directly available.

A simplified strike calendar was therefore manually created
to enable monthly aggregation and integration
with the Transilien punctuality dataset.
"""

'\nStrike dates were manually compiled using public SNCF strike records\nand historical railway strike timelines available on Wikipedia:\nhttps://wikimonde.com/article/Liste_des_gr%C3%A8ves_%C3%A0_la_SNCF\n\nAn official structured dataset containing daily strike observations\nfor the required analysis period was not directly available.\n\nA simplified strike calendar was therefore manually created\nto enable monthly aggregation and integration\nwith the Transilien punctuality dataset.\n'

In [71]:
"""
We load SNCF strike observations
to enrich the Transilien punctuality analysis.
"""

greves = pd.read_csv(
    "../Data/raw/greves_sncf.csv"
)

greves.info()
greves.head()

<class 'pandas.DataFrame'>
RangeIndex: 123 entries, 0 to 122
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   date    123 non-null    str  
dtypes: str(1)
memory usage: 1.1 KB


,date
0,2013-06-13
1,2013-09-10
2,2013-12-11
3,2014-06-09
4,2014-06-10


In [72]:
## Strike dataset preprocessing

In [73]:
"""
We convert the date column
into datetime format
"""
greves["date"] = pd.to_datetime(
    greves["date"]
)
greves.info()
greves.head()

<class 'pandas.DataFrame'>
RangeIndex: 123 entries, 0 to 122
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    123 non-null    datetime64[us]
dtypes: datetime64[us](1)
memory usage: 1.1 KB


,date
0,2013-06-13
1,2013-09-10
2,2013-12-11
3,2014-06-09
4,2014-06-10


In [74]:
# Verify strike date coverage
print(greves["date"].min())

print(greves["date"].max())

2013-06-13 00:00:00
2024-12-25 00:00:00


In [75]:
"""
We create temporal variables
for monthly strike aggregation.
"""

greves["annee"] = greves["date"].dt.year

greves["mois"] = greves["date"].dt.month

greves.head()

,date,annee,mois
0,2013-06-13,2013,6
1,2013-09-10,2013,9
2,2013-12-11,2013,12
3,2014-06-09,2014,6
4,2014-06-10,2014,6


In [76]:
## Monthly strike aggregation

In [77]:
"""
We aggregate strike observations
at monthly level.
"""

greves_mensuelles = (
    greves
    .groupby(["annee", "mois"], as_index=False)
    .size()
    .rename(columns={
        "size": "nb_jours_greve"
    })
)

greves_mensuelles.head()

,annee,mois,nb_jours_greve
0,2013,6,1
1,2013,9,1
2,2013,12,1
3,2014,6,11
4,2016,5,3


In [78]:
## Strike and Transilien merge

In [79]:
"""
We merge monthly strike indicators
with the Transilien dataset.
"""

df = df.merge(
    greves_mensuelles,
    on=["annee", "mois"],
    how="left"
)

df.head(20)

,date,service,ligne,nom_de_la_ligne,taux_de_ponctualite,nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard,annee,mois,nom_mois,trimestre,taux_irregularite,nb_jours_feries_mois,saison,temperature_moyenne,pluie_mm,vent_moyen,nb_jours_greve
0,2013-01-01,RER,A,RER A,83.6,5.1,2013,1,January,1,16.4,1,Hiver,4.174194,42.2,2.800000,NaN
1,2013-01-01,Transilien,R,Paris Sud Est,87.2,6.8,2013,1,January,1,12.8,1,Hiver,4.174194,42.2,2.800000,NaN
2,2013-03-01,Transilien,H,Paris Nord Ouest,92.3,12.0,2013,3,March,1,7.7,0,Printemps,5.545161,36.6,3.106452,NaN
3,2013-04-01,Transilien,N,Paris Montparnasse,90.2,9.2,2013,4,April,2,9.8,1,Printemps,11.040000,29.7,3.363333,NaN
4,2013-05-01,RER,D,RER D,87.1,6.8,2013,5,May,2,12.9,4,Printemps,12.587097,111.2,2.861290,NaN
5,2013-05-01,Transilien,K,Paris Nord Crépy,81.3,4.3,2013,5,May,2,18.7,4,Printemps,12.587097,111.2,2.861290,NaN
6,2013-06-01,Transilien,N,Paris Montparnasse,93.9,15.4,2013,6,June,2,6.1,0,Été,17.370000,64.0,3.026667,1.0
7,2013-06-01,Transilien,P,Paris Est,89.4,8.4,2013,6,June,2,10.6,0,Été,17.370000,64.0,3.026667,1.0
8,2013-08-01,Transilien,P,Paris Est,92.1,11.7,2013,8,August,3,7.9,1,Été,20.554839,37.6,2.438710,NaN
9,2013-09-01,RER,A,RER A,83.3,5.0,2013,9,September,3,16.7,0,Automne,17.213333,49.5,2.280000,1.0


In [80]:
## Strike feature validation

In [81]:
"""
Months without strikes are replaced by 0.
"""

df["nb_jours_greve"] = (
    df["nb_jours_greve"]
    .fillna(0)
)

df["nb_jours_greve"].value_counts()

nb_jours_greve
0.0     1465
1.0      142
2.0       90
5.0       26
12.0      26
3.0       13
8.0       13
6.0       13
16.0      13
11.0      13
Name: count, dtype: int64

In [82]:
""" Verify missing values after strike merge
"""
df["nb_jours_greve"].isna().sum()

np.int64(0)

In [83]:
"""
Analyze strike feature distribution
"""
df["nb_jours_greve"].describe()

count    1814.000000
mean        0.736494
std         2.375417
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        16.000000
Name: nb_jours_greve, dtype: float64

In [84]:
"""
Display strike feature
"""
df[
    [
        "date",
        "ligne",
        "nb_jours_greve"
    ]
].head(30)

,date,ligne,nb_jours_greve
0,2013-01-01,A,0.0
1,2013-01-01,R,0.0
2,2013-03-01,H,0.0
3,2013-04-01,N,0.0
4,2013-05-01,D,0.0
5,2013-05-01,K,0.0
6,2013-06-01,N,1.0
7,2013-06-01,P,1.0
8,2013-08-01,P,0.0
9,2013-09-01,A,1.0


In [85]:
"""
We identify the months with the highest number of strike days
to validate the strike enrichment feature
and verify the consistency of major social movements over time.
"""
df[
    [
        "annee",
        "mois",
        "nb_jours_greve"
    ]
].drop_duplicates().sort_values(
    "nb_jours_greve",
    ascending=False
).head()

,annee,mois,nb_jours_greve
285,2023,3,16.0
146,2018,4,12.0
148,2018,5,12.0
402,2014,6,11.0
149,2018,6,8.0


In [86]:
## Additional KPI engineering

In [87]:
"""
We categorize punctuality levels
to simplify BI analysis and dashboard visualization.
"""

df["categorie_ponctualite"] = pd.cut(
    df["taux_de_ponctualite"],
    bins=[0, 70, 80, 90, 95, 100],
    labels=[
        "Critical",
        "Bad",
        "Medium",
        "Good",
        "Excellent"
    ]
)

In [88]:
## Final dataset validation

In [89]:
"""
Display final dataset structure
"""
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1814 entries, 0 to 1813
Data columns (total 18 columns):
 #   Column                                                    Non-Null Count  Dtype         
---  ------                                                    --------------  -----         
 0   date                                                      1814 non-null   datetime64[us]
 1   service                                                   1814 non-null   str           
 2   ligne                                                     1814 non-null   str           
 3   nom_de_la_ligne                                           1814 non-null   str           
 4   taux_de_ponctualite                                       1813 non-null   float64       
 5   nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard  1814 non-null   float64       
 6   annee                                                     1814 non-null   int32         
 7   mois                                                 

In [90]:
"""
Display final dataset columns
"""
df.columns

Index(['date', 'service', 'ligne', 'nom_de_la_ligne', 'taux_de_ponctualite',
       'nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard', 'annee',
       'mois', 'nom_mois', 'trimestre', 'taux_irregularite',
       'nb_jours_feries_mois', 'saison', 'temperature_moyenne', 'pluie_mm',
       'vent_moyen', 'nb_jours_greve', 'categorie_ponctualite'],
      dtype='str')

In [91]:
"""
Analyze missing values in the final dataset
"""
df.isna().sum()

date                                                        0
service                                                     0
ligne                                                       0
nom_de_la_ligne                                             0
taux_de_ponctualite                                         1
nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard    0
annee                                                       0
mois                                                        0
nom_mois                                                    0
trimestre                                                   0
taux_irregularite                                           1
nb_jours_feries_mois                                        0
saison                                                      0
temperature_moyenne                                         0
pluie_mm                                                    0
vent_moyen                                                  0
nb_jours

In [92]:
"""
We identify observations with missing punctuality values
to validate the consistency and completeness
of the final Transilien dataset.
"""
df[
    df["taux_de_ponctualite"].isna()
]

,date,service,ligne,nom_de_la_ligne,taux_de_ponctualite,nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard,annee,mois,nom_mois,trimestre,taux_irregularite,nb_jours_feries_mois,saison,temperature_moyenne,pluie_mm,vent_moyen,nb_jours_greve,categorie_ponctualite
764,2014-06-01,RER,B,RER B,NaN,0.0,2014,6,June,2,NaN,1,Été,18.563333,86.0,3.036667,11.0,NaN


In [93]:
"""
We remove observations with missing punctuality values
to ensure the consistency and quality
of the final analytical dataset.
"""

df = df.dropna(
    subset=["taux_de_ponctualite"]
)

In [94]:
"""
Analyze missing values in the final dataset
"""
df.isna().sum()

date                                                        0
service                                                     0
ligne                                                       0
nom_de_la_ligne                                             0
taux_de_ponctualite                                         0
nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard    0
annee                                                       0
mois                                                        0
nom_mois                                                    0
trimestre                                                   0
taux_irregularite                                           0
nb_jours_feries_mois                                        0
saison                                                      0
temperature_moyenne                                         0
pluie_mm                                                    0
vent_moyen                                                  0
nb_jours

In [95]:
## Final enrichment conclusion

In [96]:
"""
We export the final enriched dataset
for future BI analysis and dashboard creation.
"""

df.to_csv(
    "../Data/processed/transilien_enriched.csv",
    index=False,
    encoding="utf-8-sig"
)

In [97]:
## ANNUAL KPI DATASET 

In [114]:
""" 
We create an aggregated annual KPI dataset
to simplify dashboard creation in SAP Analytics Cloud.

This aggregation avoids duplicated calculations
caused by the presence of multiple railway lines
and monthly observations in the original dataset.

The resulting dataset provides a global yearly view
of the Transilien network performance.
"""

df_kpi_year = (
    df.groupby("annee", as_index=False)
    .agg({
        "taux_de_ponctualite": "mean",
        "taux_irregularite": "mean",
        "temperature_moyenne": "mean",
        "pluie_mm": "mean",
        "vent_moyen": "mean",
    })
)


In [115]:
greve_year = (
    greves.groupby("annee")
    .size()
    .reset_index(name="nb_jours_greve")
)
greve_year

,annee,nb_jours_greve
0,2013,3
1,2014,11
2,2016,8
3,2018,32
4,2019,16
5,2020,4
6,2022,13
7,2023,21
8,2024,15


In [120]:
df_kpi_year = df_kpi_year.merge(
    greve_year,
    on="annee",
    how="left"
)
df_kpi_year["greve_year"] = (
    df_kpi_year["nb_jours_greve"]
    .fillna(0)
)

In [121]:
## Testing the annual KPI dataset

In [123]:
df_kpi_year.drop(columns=["nb_jours_greve"], inplace=True)
df_kpi_year.drop(columns=["nb_jours_greve_x"], inplace=True)
df_kpi_year.drop(columns=["nb_jours_greve_y"], inplace=True)
df_kpi_year.head()

,annee,taux_de_ponctualite,taux_irregularite,temperature_moyenne,pluie_mm,vent_moyen,greve_year
0,2013,87.830128,12.169872,11.913766,51.358333,2.980319,3.0
1,2014,88.615194,11.384806,13.277799,59.513548,2.878116,11.0
2,2015,88.298321,11.701679,13.149092,42.208333,2.993870,0.0
3,2016,88.144167,11.855833,12.539165,54.816667,2.882261,8.0
4,2017,87.881263,12.118737,13.038703,61.591667,2.867943,0.0


In [124]:

print(df_kpi_year.info())
print(df_kpi_year.isna().sum())
df_kpi_year.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   annee                12 non-null     int32  
 1   taux_de_ponctualite  12 non-null     float64
 2   taux_irregularite    12 non-null     float64
 3   temperature_moyenne  12 non-null     float64
 4   pluie_mm             12 non-null     float64
 5   vent_moyen           12 non-null     float64
 6   greve_year           12 non-null     float64
dtypes: float64(6), int32(1)
memory usage: 756.0 bytes
None
annee                  0
taux_de_ponctualite    0
taux_irregularite      0
temperature_moyenne    0
pluie_mm               0
vent_moyen             0
greve_year             0
dtype: int64


np.int64(0)

In [125]:
df_kpi_year.describe()

,annee,taux_de_ponctualite,taux_irregularite,temperature_moyenne,pluie_mm,vent_moyen,greve_year
count,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000
mean,2018.500000,89.542360,10.457640,13.279207,56.745365,3.016358,10.250000
std,3.605551,1.520270,1.520270,0.713736,8.533496,0.131348,9.836158
min,2013.000000,87.830128,8.108564,11.913766,42.208333,2.867943,0.000000
25%,2015.750000,88.259782,9.326381,12.913818,52.385038,2.918173,2.250000
50%,2018.500000,89.241513,10.758487,13.252439,58.052778,2.987094,9.500000
75%,2021.250000,90.673619,11.740218,13.916469,60.858333,3.064872,15.250000
max,2024.000000,91.891436,12.169872,14.286769,75.091667,3.257846,32.000000


In [128]:
df_kpi_year[
    ["annee", "greve_year"]
]

,annee,greve_year
0,2013,3.0
1,2014,11.0
2,2015,0.0
3,2016,8.0
4,2017,0.0
5,2018,32.0
6,2019,16.0
7,2020,4.0
8,2021,0.0
9,2022,13.0


In [129]:
"""
Export annual KPI dataset
for SAP Analytics Cloud dashboards.
"""

df_kpi_year.to_csv(
    "../Data/processed/kpi_annuel.csv",
    index=False
)

In [ ]:
## Seasonal ANALYSIS DATASET

In [131]:
"""
To improve the reliability of seasonal analysis
within SAP Analytics Cloud, a dedicated seasonal dataset
is created using pre-aggregated indicators.

Instead of relying entirely on SAC aggregations,
KPIs are aggregated directly in Python
to ensure cleaner analytical granularity
and more reliable comparisons between seasons.

Each row represents
the average network performance for a given season.
"""

'\nTo improve the reliability of seasonal analysis\nwithin SAP Analytics Cloud, a dedicated seasonal dataset\nis created using pre-aggregated indicators.\n\nInstead of relying entirely on SAC aggregations,\nKPIs are aggregated directly in Python\nto ensure cleaner analytical granularity\nand more reliable comparisons between seasons.\n\nEach row represents\nthe average network performance for a given season.\n'

In [132]:
"""
Creation of the seasonal aggregated dataset.
"""

df_saison = (
    df.groupby("saison", as_index=False)
    .agg({
        "taux_de_ponctualite": "mean",
        "taux_irregularite": "mean",
        "temperature_moyenne": "mean",
        "pluie_mm": "mean",
        "vent_moyen": "mean"
    })
)

df_saison.head()

,saison,taux_de_ponctualite,taux_irregularite,temperature_moyenne,pluie_mm,vent_moyen
0,Automne,88.647962,11.352038,13.453728,56.727778,2.775227
1,Hiver,88.953520,11.046480,6.357500,55.257273,3.352010
2,Printemps,90.544554,9.455446,12.009568,59.807973,3.163379
3,Été,89.940702,10.059298,20.723516,55.367811,2.783217


In [133]:
## WEATHER ANALYSIS DATASET

In [134]:
"""
To perform reliable weather-related analysis
within SAP Analytics Cloud, a dedicated daily
weather dataset is created.

The objective is to analyze potential relationships
between weather conditions and network performance.

The dataset is aggregated at the DAILY level
to preserve weather variability and avoid
incorrect aggregations caused by multiple railway lines
or duplicated monthly observations.

Each row represents:
- one day,
- average weather conditions,
- and average network performance indicators.
"""

'\nTo perform reliable weather-related analysis\nwithin SAP Analytics Cloud, a dedicated daily\nweather dataset is created.\n\nThe objective is to analyze potential relationships\nbetween weather conditions and network performance.\n\nThe dataset is aggregated at the DAILY level\nto preserve weather variability and avoid\nincorrect aggregations caused by multiple railway lines\nor duplicated monthly observations.\n\nEach row represents:\n- one day,\n- average weather conditions,\n- and average network performance indicators.\n'

In [144]:
df_meteo = (
    df.groupby(
        ["annee", "mois", "nom_mois"],
        as_index=False
    )
    .agg({
        "pluie_mm": "mean",
        "temperature_moyenne": "mean",
        "vent_moyen": "mean",
        "taux_irregularite": "mean",
        "taux_de_ponctualite": "mean"
    })
)
df_meteo.head()

,annee,mois,nom_mois,pluie_mm,temperature_moyenne,vent_moyen,taux_irregularite,taux_de_ponctualite
0,2013,1,January,42.2,4.174194,2.800000,13.684615,86.315385
1,2013,2,February,37.4,3.321429,3.492857,12.846154,87.153846
2,2013,3,March,36.6,5.545161,3.106452,13.900000,86.100000
3,2013,4,April,29.7,11.040000,3.363333,11.861538,88.138462
4,2013,5,May,111.2,12.587097,2.861290,9.276923,90.723077


In [145]:
"""
Checking rainfall value ranges
to ensure realistic precipitation values.
"""

print(df_meteo["pluie_mm"].min())
print(df_meteo["pluie_mm"].max())

1.6
178.59999999999997


In [146]:
"""
Verification of missing values.
"""

df_meteo.isna().sum()

annee                  0
mois                   0
nom_mois               0
pluie_mm               0
temperature_moyenne    0
vent_moyen             0
taux_irregularite      0
taux_de_ponctualite    0
dtype: int64

In [147]:
"""
Export of the weather analysis dataset
for SAP Analytics Cloud dashboards.
"""

df_meteo.to_csv(
    "../Data/processed/meteo_analysis.csv",
    index=False
)

In [148]:
"""
========================================================
ANALYTICAL INTERPRETATION
========================================================

This dataset supports exploratory analysis
of weather impacts on network performance.

It will be used to create:
- rainfall vs irregularity scatter plots,
- temperature vs punctuality analysis,
- and weather-related BI visualizations
within SAP Analytics Cloud.
"""

'\n========================================================\nANALYTICAL INTERPRETATION\n========================================================\n\nThis dataset supports exploratory analysis\nof weather impacts on network performance.\n\nIt will be used to create:\n- rainfall vs irregularity scatter plots,\n- temperature vs punctuality analysis,\n- and weather-related BI visualizations\nwithin SAP Analytics Cloud.\n'